In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-04-07 14:09:21 +05:30)


In [2]:
import tensorflow as tf

import os
from pathlib import Path

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau

import numpy as np
import cv2

time: 22.3 s (started: 2026-04-07 14:09:21 +05:30)


In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 0 ns (started: 2026-04-07 14:09:43 +05:30)


In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True
time: 16 ms (started: 2026-04-07 14:09:43 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 16 ms (started: 2026-04-07 14:09:43 +05:30)


In [6]:
bad_files = (
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpeg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.png"))
)

deleted = 0

for f in bad_files:
    try:
        f.unlink()
        deleted += 1
    except Exception as e:
        print(f"Error deleting {f}: {e}")

print(f"Deleted {deleted} augmented files")

Deleted 0 augmented files
time: 78 ms (started: 2026-04-07 14:09:43 +05:30)


In [7]:
pneumonia_count = len(list((TRAIN_DIR / "PNEUMONIA").glob("*")))
OVERSAMPLE_TARGET = pneumonia_count

aug_gen = ImageDataGenerator(
    rotation_range=10,            
    width_shift_range=0.1,       
    height_shift_range=0.1,       
    zoom_range=0.2,               
    horizontal_flip=True,         
    brightness_range=[0.8, 1.2],  
    shear_range=0.1,              
    fill_mode='nearest'          
)

normal_dir = TRAIN_DIR / "NORMAL"

existing_files = (
    list(normal_dir.glob("*.jpeg")) +
    list(normal_dir.glob("*.jpg"))  +
    list(normal_dir.glob("*.png"))
)

current_count = len(existing_files)
print(f"Current NORMAL count: {current_count}")

if current_count >= OVERSAMPLE_TARGET:
    print(f"Already at or above target ({OVERSAMPLE_TARGET}). Skipping oversampling.")

else:
    source_files = [f for f in existing_files if not f.stem.startswith("aug_")]

    if len(source_files) == 0:
        raise ValueError("No original images available for augmentation.")

    needed = OVERSAMPLE_TARGET - current_count
    print(f"Need to generate {needed} augmented images...")

    generated = 0

    start_idx = len(existing_files)

    while generated < needed:
        for source_path in source_files:
            if generated >= needed:
                break

            img = cv2.imread(str(source_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            img = cv2.resize(img, IMG_SIZE)

            img_array = img.reshape(1, IMG_SIZE[0], IMG_SIZE[1], 1).astype("float32")
            img_array /= 255.0

            aug_iterator = aug_gen.flow(img_array, batch_size=1)
            aug_image = next(aug_iterator)[0]

            aug_image = (aug_image * 255).clip(0, 255).astype("uint8")
            aug_image = aug_image.reshape(IMG_SIZE[0], IMG_SIZE[1])

            save_path = normal_dir / f"aug_{start_idx + generated:05d}.jpeg"
            cv2.imwrite(str(save_path), aug_image)

            generated += 1

    print(f"Generated {generated} augmented images.")

    final_count = len(
        list(normal_dir.glob("*.jpeg")) +
        list(normal_dir.glob("*.jpg"))  +
        list(normal_dir.glob("*.png"))
    )

    pneumonia_count = len(list((TRAIN_DIR / "PNEUMONIA").glob("*")))

    print(f"\nNORMAL count after oversampling : {final_count}")
    print(f"PNEUMONIA count (unchanged)     : {pneumonia_count}")

    ratio = round(final_count / pneumonia_count, 2)
    print(f"New ratio (Normal:Pneumonia)    : {ratio}:1")

Current NORMAL count: 1199
Need to generate 2281 augmented images...
Generated 2281 augmented images.

NORMAL count after oversampling : 3480
PNEUMONIA count (unchanged)     : 3480
New ratio (Normal:Pneumonia)    : 1.0:1
time: 48.3 s (started: 2026-04-07 14:09:44 +05:30)


In [8]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

time: 0 ns (started: 2026-04-07 14:10:32 +05:30)


In [9]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 16 ms (started: 2026-04-07 14:10:32 +05:30)


In [10]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

Found 6960 images belonging to 2 classes.
time: 563 ms (started: 2026-04-07 14:10:32 +05:30)


In [11]:
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=False
)

Found 553 images belonging to 2 classes.
time: 94 ms (started: 2026-04-07 14:10:33 +05:30)


In [12]:
from tensorflow.keras.optimizers import Adam

def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),

        layers.GlobalAveragePooling2D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return m

time: 15 ms (started: 2026-04-07 14:10:33 +05:30)


layers.GlobalAveragePooling2D() :-
To reduce parameters and prevent overfitting by summarizing each feature map instead of flattening everything

In [13]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',     
        patience=6,
        restore_best_weights=True,
        mode='min'             
    ),
    ReduceLROnPlateau(
        monitor='val_loss',     
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        mode='min'             
    ),
    ModelCheckpoint(
        "best_model_OverSampling.h5",
        monitor='val_loss',      
        save_best_only=True,
        mode='min' 
    )
]

time: 0 ns (started: 2026-04-07 14:10:33 +05:30)


In [14]:
model = build_model()

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 871ms/step - accuracy: 0.7861 - loss: 0.4490

218/218 ━━━━━━━━━━━━━━━━━━━━ 202s 907ms/step - accuracy: 0.8257 - loss: 0.3734 - val_accuracy: 0.7288 - val_loss: 1.3657 - learning_rate: 1.0000e-04
Epoch 2/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 195s 892ms/step - accuracy: 0.8907 - loss: 0.2616 - val_accuracy: 0.7288 - val_loss: 3.1621 - learning_rate: 1.0000e-04
Epoch 3/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 193s 881ms/step - accuracy: 0.9168 - loss: 0.2268 - val_accuracy: 0.7288 - val_loss: 1.5015 - learning_rate: 1.0000e-04
Epoch 4/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 857ms/step - accuracy: 0.9269 - loss: 0.1966

218/218 ━━━━━━━━━━━━━━━━━━━━ 193s 882ms/step - accuracy: 0.9267 - loss: 0.2011 - val_accuracy: 0.8897 - val_loss: 0.2822 - learning_rate: 1.0000e-04
Epoch 5/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 192s 881ms/step - accuracy: 0.9335 - loss: 0.1951 - val_accuracy: 0.5425 - val_loss: 1.0337 - learning_rate: 1.0000e-04
Epoch 6/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 215s 987ms/step - accuracy: 0.9351 - loss: 0.1981 - val_accuracy: 0.7306 - val_loss: 0.4194 - learning_rate: 1.0000e-04
Epoch 7/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 196s 896ms/step - accuracy: 0.9398 - loss: 0.1826 - val_accuracy: 0.8915 - val_loss: 0.2887 - learning_rate: 1.0000e-04
Epoch 8/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 197s 900ms/step - accuracy: 0.9405 - loss: 0.1731 - val_accuracy: 0.8626 - val_loss: 0.3336 - learning_rate: 3.0000e-05
Epoch 9/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 194s 891ms/step - accuracy: 0.9425 - loss: 0.1736 - val_accuracy: 0.8662 - val_loss: 0.3370 - learning_rate: 3.0000e-05
Epoch 10/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 864ms/s

218/218 ━━━━━━━━━━━━━━━━━━━━ 195s 893ms/step - accuracy: 0.9417 - loss: 0.1730 - val_accuracy: 0.9168 - val_loss: 0.2489 - learning_rate: 3.0000e-05
Epoch 11/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 199s 912ms/step - accuracy: 0.9435 - loss: 0.1658 - val_accuracy: 0.8318 - val_loss: 0.3434 - learning_rate: 3.0000e-05
Epoch 12/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 218s 985ms/step - accuracy: 0.9473 - loss: 0.1644 - val_accuracy: 0.7595 - val_loss: 0.4569 - learning_rate: 3.0000e-05
Epoch 13/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 199s 911ms/step - accuracy: 0.9467 - loss: 0.1638 - val_accuracy: 0.8879 - val_loss: 0.2789 - learning_rate: 3.0000e-05
Epoch 14/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 204s 934ms/step - accuracy: 0.9455 - loss: 0.1602 - val_accuracy: 0.8083 - val_loss: 0.3849 - learning_rate: 9.0000e-06
Epoch 15/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 882ms/step - accuracy: 0.9495 - loss: 0.1532

218/218 ━━━━━━━━━━━━━━━━━━━━ 198s 906ms/step - accuracy: 0.9474 - loss: 0.1594 - val_accuracy: 0.9259 - val_loss: 0.2454 - learning_rate: 9.0000e-06
Epoch 16/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 195s 893ms/step - accuracy: 0.9486 - loss: 0.1553 - val_accuracy: 0.7722 - val_loss: 0.4268 - learning_rate: 9.0000e-06
Epoch 17/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.9517 - loss: 0.1529 - val_accuracy: 0.8897 - val_loss: 0.2887 - learning_rate: 9.0000e-06
Epoch 18/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 195s 894ms/step - accuracy: 0.9511 - loss: 0.1538 - val_accuracy: 0.8481 - val_loss: 0.3486 - learning_rate: 9.0000e-06
Epoch 19/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 195s 894ms/step - accuracy: 0.9514 - loss: 0.1531 - val_accuracy: 0.8463 - val_loss: 0.3422 - learning_rate: 2.7000e-06
Epoch 20/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 196s 898ms/step - accuracy: 0.9539 - loss: 0.1467 - val_accuracy: 0.8807 - val_loss: 0.3040 - learning_rate: 2.7000e-06
Epoch 21/25
218/218 ━━━━━━━━━━━━━━━━━━━━ 195s 894

In [15]:
bad_files = (
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpeg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.png"))
)

deleted = 0

for f in bad_files:
    try:
        f.unlink()
        deleted += 1
    except Exception as e:
        print(f"Error deleting {f}: {e}")

print(f"Deleted {deleted} augmented files")

Deleted 2281 augmented files
time: 890 ms (started: 2026-04-07 15:20:18 +05:30)
